In [ ]:
#%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.6/22.6 MB 11.0 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 10.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.9/742.9 kB 8.4 MB/s  0:00:00
  Attempting uninstall: protobuf90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  6/27 [pybase64]
    Found existing installation: protobuf 7.35.0━━━━━━━━━━━━━━  6/27 [pybase64]
    Uninstalling protobuf-7.35.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  6/27 [pybase64]
      Successfully uninstalled protobuf-7.35.0━━━━━━━━━━━━━━━━  6/27 [pybase64]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27/27 [chromadb]/27 [chromadb]s]ry-sdk]protos]ons]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import json

with open("knowledge_base.json", "r") as f:
    kb = json.load(f)

In [2]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="knowledge_base"
)

In [3]:
collection.add(
    documents=[item["text"] for item in kb],
    metadatas=[
        {"source": item["source"]}
        for item in kb
    ],
    ids=[item["id"] for item in kb]
)

In [ ]:
from google import genai
import os

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

In [9]:
def retrieve(question, k=3):

    results = collection.query(
        query_texts=[question],
        n_results=k
    )

    return results

In [10]:
def build_prompt(question):

    results = retrieve(question)

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    context = ""

    for doc, meta in zip(docs, metas):
        context += (
            f"[SOURCE: {meta['source']}]\n"
            f"{doc}\n\n"
        )

    prompt = f"""
You are a retrieval-augmented QA assistant.

Answer ONLY using the provided context.

If the answer cannot be found in the context,
respond exactly:

I don't know.

Context:
{context}

Question:
{question}

Include source citations in your answer.
"""

    return prompt, results

In [21]:
def answer_question(question):

    prompt, results = build_prompt(question)

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    return response.text, results

In [22]:
questions = [
    "How long do I have to get a full refund?",
    "How do I reset my password?",
    "What is the company's stock price today?"
]

for question in questions:

    answer, results = answer_question(question)

    print("=" * 60)
    print("QUESTION:")
    print(question)

    print("\nRETRIEVED SOURCES:")

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    for doc, meta in zip(docs, metas):
        print(f"- {meta['source']}")
        print(doc)

    print("\nFINAL ANSWER:")
    print(answer)
    print()

QUESTION:
How long do I have to get a full refund?

RETRIEVED SOURCES:
- policy.md
Our refund policy allows a full refund within 30 days of purchase, provided the item is unused and in its original packaging. After 30 days, only store credit is offered.
- policy.md
To cancel your subscription, open Account Settings and choose End Plan. Cancellation takes effect at the end of the current billing period; no partial refunds are given for the remaining days.
- policy.md
Premium plan members get priority support, with a guaranteed first response within four business hours. Standard members receive a response within two business days.

FINAL ANSWER:
You have 30 days from the date of purchase to get a full refund [SOURCE: policy.md].

QUESTION:
How do I reset my password?

RETRIEVED SOURCES:
- it.md
Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.
- handbook.md
To power up a dev